In [1]:
!pip install datasets evaluate --upgrade
# !python -m spacy download en-core-web-sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 10.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 6.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.31.0
    Uninstalling requests-2.31.0:
      Successfully uninstalled requests-2.31.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. T

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import spacy
import torchtext.vocab
from tqdm import tqdm
import evaluate
from spacy.tokenizer import Tokenizer
from spacy.lang.tokenizer_exceptions import BASE_EXCEPTIONS
from spacy.util import compile_infix_regex

/usr/local/lib/python3.10/dist-packages/torchtext/vocab/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/usr/local/lib/python3.10/dist-packages/torchtext/utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# File paths
data_folder = '/content/drive/MyDrive/datasets/manipuri MT/English-Manipuri/parallel'
train_src_file = os.path.join(data_folder, 'en-mni-train-mni.txt')
train_tgt_file = os.path.join(data_folder, 'en-mni-train-en.txt')
valid_src_file = os.path.join(data_folder, 'en-mni-valid-mni.txt')
valid_tgt_file = os.path.join(data_folder, 'en-mni-valid-en.txt')
test_src_file = os.path.join(data_folder, 'en-mni-test-mni.txt')
test_tgt_file = os.path.join(data_folder, 'en-mni-test-en.txt')

In [5]:
# Load spaCy tokenizers
en_nlp = spacy.load("en_core_web_sm")

# Create a custom tokenizer for Manipuri
def custom_tokenizer(nlp):
    infix_re = compile_infix_regex(nlp.Defaults.infixes)
    return Tokenizer(nlp.vocab, rules=BASE_EXCEPTIONS, prefix_search=nlp.tokenizer.prefix_search,
                     suffix_search=nlp.tokenizer.suffix_search, infix_finditer=infix_re.finditer,
                     token_match=nlp.tokenizer.token_match, url_match=nlp.tokenizer.url_match)

# Using a blank spaCy language class for the custom tokenizer
from spacy.language import Language
mni_nlp = Language()
mni_nlp.tokenizer = custom_tokenizer(mni_nlp)


In [6]:
# Read data from files
def read_data(src_file, tgt_file):
    with open(src_file, 'r', encoding='utf-8') as src_f, open(tgt_file, 'r', encoding='utf-8') as tgt_f:
        src_lines = src_f.readlines()
        tgt_lines = tgt_f.readlines()
    return src_lines, tgt_lines

train_src_lines, train_tgt_lines = read_data(train_src_file, train_tgt_file)
valid_src_lines, valid_tgt_lines = read_data(valid_src_file, valid_tgt_file)
test_src_lines, test_tgt_lines = read_data(test_src_file, test_tgt_file)

In [8]:
# Print sample sentences
num_samples = 5
print(f"Sample Training Data (Source -> Target):")
for i in range(num_samples):
    print(f"{i+1}. {train_src_lines[i].strip()} -> {train_tgt_lines[i].strip()}")

print(f"\nSample Validation Data (Source -> Target):")
for i in range(num_samples):
    print(f"{i+1}. {valid_src_lines[i].strip()} -> {valid_tgt_lines[i].strip()}")

print(f"\nSample Test Data (Source -> Target):")
for i in range(num_samples):
    print(f"{i+1}. {test_src_lines[i].strip()} -> {test_tgt_lines[i].strip()}")

Sample Training Data (Source -> Target):
1. এনর্জি পোর্টেল অসিনা মথক্কী ৱারোলশীং অসিগী ঈ-পাউ অসি নহাক্না মখোয়বু শীজিন্ননিংগদবা অমদি মখোয়গা লোয়নরিবা কান্নবশীং লৌবা পামহন্নবগী মওংদা পুক্নিং থৌগৎনিংঙাই ওইবা ৱারিশীংগা লোয়ননা পীনবা হোৎনৈ । -> the energy portal attempts to give information on the above aspects with inspiring stories that would motivate you to use them and derive the associated benefits .
2. মহৌশাগী উপদ্রবশীংনা শোকহল্লবা লৌমীশীংদা রিলিফ পীনবগীদমক্তা ,  বেঙ্কশীংদা অহানবা চহিদুগী ওইনা রিষ্ট্রকচর তৌরবা এমাউন্ট অদুদা ২%গী ইন্টরেষ্ট সবভেন্সন পীগনি । -> to provide relief to the farmers affected by natural calamities , the interest subvention of 2 % will be provided to banks for the first year on the restructured amount .
3. প্রধানমন্ত্রী শ্রী নরেন্দ্র মোদীনা লুচিংবা কেন্দ্রগী মন্ত্রীমন্দলনা দিপার্তমেন্ত ওফ ইকোনোমীক এফিয়ার্স ( ইন্দিয়ান ইকোনোমীক সর্বিস কেদর )  অমসুং দি ত্রেজরী , গবর্নমেন্ত ওফ ওস্ত্রেলিয়াগী মরক্তা থা অহুমগী ওইনা সেকেন্দমেন্ত প্রোগ্রামগীদমক মেমোরেন্দা ওফ অন্দর্সতেন্দিং

In [9]:
# Tokenization function
def tokenize_sentences(src_lines, tgt_lines, src_nlp, tgt_nlp, max_length, lower, sos_token, eos_token):
    tokenized_data = []
    for src_line, tgt_line in zip(src_lines, tgt_lines):
        src_tokens = [token.text for token in src_nlp.tokenizer(src_line.strip())][:max_length]
        tgt_tokens = [token.text for token in tgt_nlp.tokenizer(tgt_line.strip())][:max_length]
        if lower:
            src_tokens = [token.lower() for token in src_tokens]
            tgt_tokens = [token.lower() for token in tgt_tokens]
        src_tokens = [sos_token] + src_tokens + [eos_token]
        tgt_tokens = [sos_token] + tgt_tokens + [eos_token]
        tokenized_data.append({"src_tokens": src_tokens, "tgt_tokens": tgt_tokens})
    return tokenized_data

In [10]:
# Parameters
max_length = 1000
lower = True
sos_token = "<sos>"
eos_token = "<eos>"

train_data = tokenize_sentences(train_src_lines, train_tgt_lines, mni_nlp, en_nlp, max_length, lower, sos_token, eos_token)
valid_data = tokenize_sentences(valid_src_lines, valid_tgt_lines, mni_nlp, en_nlp, max_length, lower, sos_token, eos_token)
test_data = tokenize_sentences(test_src_lines, test_tgt_lines, mni_nlp, en_nlp, max_length, lower, sos_token, eos_token)

In [11]:
# Build vocabularies
min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"

special_tokens = [unk_token, pad_token, sos_token, eos_token]

src_vocab = torchtext.vocab.build_vocab_from_iterator(
    (example["src_tokens"] for example in train_data),
    min_freq=min_freq,
    specials=special_tokens,
)

tgt_vocab = torchtext.vocab.build_vocab_from_iterator(
    (example["tgt_tokens"] for example in train_data),
    min_freq=min_freq,
    specials=special_tokens,
)


In [12]:
# Ensure special tokens are the same for both vocabularies
assert src_vocab[unk_token] == tgt_vocab[unk_token]
assert src_vocab[pad_token] == tgt_vocab[pad_token]

unk_index = src_vocab[unk_token]
pad_index = src_vocab[pad_token]

src_vocab.set_default_index(unk_index)
tgt_vocab.set_default_index(unk_index)

In [13]:
# Numericalize examples
def numericalize_examples(data, src_vocab, tgt_vocab):
    numericalized_data = []
    for example in data:
        src_ids = src_vocab.lookup_indices(example["src_tokens"])
        tgt_ids = tgt_vocab.lookup_indices(example["tgt_tokens"])
        numericalized_data.append({"src_ids": src_ids, "tgt_ids": tgt_ids})
    return numericalized_data

train_data = numericalize_examples(train_data, src_vocab, tgt_vocab)
valid_data = numericalize_examples(valid_data, src_vocab, tgt_vocab)
test_data = numericalize_examples(test_data, src_vocab, tgt_vocab)

In [ ]:
# # Set format
# data_type = "torch"
# format_columns = ["en_ids", "de_ids"]

# train_data = train_data.with_format(type=data_type, columns=format_columns, output_all_columns=True)
# valid_data = valid_data.with_format(type=data_type, columns=format_columns, output_all_columns=True)
# test_data = test_data.with_format(type=data_type, columns=format_columns, output_all_columns=True)


In [14]:
# Convert to Dataset objects
class TranslationDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

train_dataset = TranslationDataset(train_data)
valid_dataset = TranslationDataset(valid_data)
test_dataset = TranslationDataset(test_data)

# DataLoader and collate function
def collate_fn(batch):
    batch_src_ids = [torch.tensor(example["src_ids"]) for example in batch]
    batch_tgt_ids = [torch.tensor(example["tgt_ids"]) for example in batch]
    batch_src_ids = nn.utils.rnn.pad_sequence(batch_src_ids, padding_value=pad_index)
    batch_tgt_ids = nn.utils.rnn.pad_sequence(batch_tgt_ids, padding_value=pad_index)
    batch = {"src_ids": batch_src_ids, "tgt_ids": batch_tgt_ids}
    return batch

def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

batch_size = 128

train_data_loader = get_data_loader(train_dataset, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_dataset, batch_size, pad_index)
test_data_loader = get_data_loader(test_dataset, batch_size, pad_index)

In [15]:
# Building Model
# Encoder
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        return hidden, cell

# Decoder
class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden, cell

# Seq2Seq Model
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        assert encoder.hidden_dim == decoder.hidden_dim, "Hidden dimensions of encoder and decoder must be equal!"
        assert encoder.n_layers == decoder.n_layers, "Encoder and decoder must have equal number of layers!"

    def forward(self, src, trg, teacher_forcing_ratio):
        batch_size = trg.shape[1]
        trg_length = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)
        input = trg[0, :]
        for t in range(1, trg_length):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[t] if teacher_force else top1
        return outputs

In [16]:
# Model hyperparameters and initialization
input_dim = len(src_vocab)
output_dim = len(tgt_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
n_layers = 2
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder(input_dim, encoder_embedding_dim, hidden_dim, n_layers, encoder_dropout)
decoder = Decoder(output_dim, decoder_embedding_dim, hidden_dim, n_layers, decoder_dropout)
model = Seq2Seq(encoder, decoder, device).to(device)


In [17]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)

model.apply(init_weights)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"The model has {count_parameters(model):,} trainable parameters")

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

The model has 20,747,093 trainable parameters


In [18]:
# Training function
def train_fn(model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device):
    model.train()
    epoch_loss = 0
    for i, batch in enumerate(data_loader):
        src = batch["src_ids"].to(device)
        trg = batch["tgt_ids"].to(device)
        optimizer.zero_grad()
        output = model(src, trg, teacher_forcing_ratio)
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        trg = trg[1:].view(-1)
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(data_loader)

In [19]:
# Evaluation function
def evaluate_fn(model, data_loader, criterion, device):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            src = batch["src_ids"].to(device)
            trg = batch["tgt_ids"].to(device)
            output = model(src, trg, 0)
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            trg = trg[1:].view(-1)
            loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(data_loader)


In [20]:
# Train the model
n_epochs = 15
clip = 1
teacher_forcing_ratio = 0.5
best_valid_loss = float('inf')

for epoch in range(n_epochs):
    train_loss = 0
    model.train()
    with tqdm(total=len(train_data_loader), desc=f"Epoch {epoch+1}/{n_epochs}", unit="batch") as pbar:
        for batch in train_data_loader:
            src = batch["src_ids"].to(device)
            trg = batch["tgt_ids"].to(device)
            optimizer.zero_grad()
            output = model(src, trg, teacher_forcing_ratio)
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            trg = trg[1:].view(-1)
            loss = criterion(output, trg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
            train_loss += loss.item()
            pbar.set_postfix({"loss": loss.item()})
            pbar.update(1)
    train_loss /= len(train_data_loader)

    valid_loss = evaluate_fn(model, valid_data_loader, criterion, device)
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'best-model-2.pt')

    print(f"Epoch: {epoch+1:02}")
    print(f"\tTrain Loss: {train_loss:.3f} | Train PPL: {np.exp(train_loss):7.3f}")
    print(f"\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {np.exp(valid_loss):7.3f}")


Epoch 1/15: 100%|██████████| 170/170 [02:23<00:00,  1.19batch/s, loss=6.4]


Epoch: 01
	Train Loss: 6.679 | Train PPL: 795.452
	 Val. Loss: 6.459 |  Val. PPL: 638.557


Epoch 2/15: 100%|██████████| 170/170 [02:21<00:00,  1.20batch/s, loss=6.23]


Epoch: 02
	Train Loss: 6.334 | Train PPL: 563.301
	 Val. Loss: 6.704 |  Val. PPL: 815.281


Epoch 3/15: 100%|██████████| 170/170 [02:23<00:00,  1.19batch/s, loss=6.14]


...

Epoch: 13
	Train Loss: 4.978 | Train PPL: 145.185
	 Val. Loss: 6.025 |  Val. PPL: 413.816


Epoch 14/15: 100%|██████████| 170/170 [02:24<00:00,  1.18batch/s, loss=4.74]


Epoch: 14
	Train Loss: 4.898 | Train PPL: 133.972
	 Val. Loss: 5.992 |  Val. PPL: 400.155


Epoch 15/15: 100%|██████████| 170/170 [02:24<00:00,  1.18batch/s, loss=4.66]


Epoch: 15
	Train Loss: 4.832 | Train PPL: 125.503
	 Val. Loss: 5.951 |  Val. PPL: 384.154


In [25]:

# Load the best model
model.load_state_dict(torch.load('best-model-2.pt'))
#model.load_state_dict(torch.load('best-model.pt', map_location=torch.device('cpu')))


<All keys matched successfully>

In [26]:
# Translation function
def translate_sentence(sentence, src_vocab, trg_vocab, model, device, max_len=50):
    model.eval()
    tokens = [token.text.lower() for token in mni_nlp(sentence)]
    tokens = ["<sos>"] + tokens + ["<eos>"]
    src_indexes = [src_vocab[token] for token in tokens]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(1).to(device)
    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)
    trg_indexes = [trg_vocab["<sos>"]]
    for _ in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
        with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)
        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)
        if pred_token == trg_vocab["<eos>"]:
            break
    trg_tokens = [trg_vocab.lookup_token(i) for i in trg_indexes]
    return " ".join(trg_tokens[1:-1])  # Exclude <sos> and <eos>


In [53]:
# Translation function #debugged
def translate_sentence(sentence, src_vocab, trg_vocab, model, device, max_len=50):
    model.eval()
    # Check if the input is a tensor and decode it if necessary
    if isinstance(sentence, torch.Tensor):
        # Decode the tensor to a list of token indices
        sentence = sentence.tolist()
        # Look up each token individually and join them
        # Iterate over each token index in the sentence list and look them up individually
        sentence = " ".join([src_vocab.lookup_token(token_idx) for token_idx in sentence])
    tokens = [token.text.lower() for token in mni_nlp(sentence)]
    tokens = ["<sos>"] + tokens + ["<eos>"]
    src_indexes = [src_vocab[token] for token in tokens]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(1).to(device)
    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)
    trg_indexes = [trg_vocab["<sos>"]]
    for _ in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
        with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)
        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)
        if pred_token == trg_vocab["<eos>"]:
            break
    trg_tokens = [trg_vocab.lookup_token(i) for i in trg_indexes]
    return " ".join(trg_tokens[1:-1])  # Exclude <sos> and <eos>

In [54]:
# # BLEU score calculation
# bleu = evaluate.load("bleu")

# def calculate_bleu(data_loader, model, src_vocab, trg_vocab, device):
#     targets, predictions = [], []
#     for example in data_loader:
#         src, trg = example["src"], example["tgt"]
#         pred_trg = translate_sentence(src, src_vocab, trg_vocab, model, device)
#         targets.append([trg])  # Reference should be a list of references
#         predictions.append(pred_trg)
#     # Joining target tokens into a single string
#     targets = [[" ".join(target)] for target in targets]
#     return bleu.compute(predictions=predictions, references=targets)

In [55]:
print(next(iter(test_data_loader)))

{'src_ids': tensor([[   2,    2,    2,  ...,    2,    2,    2],
        [  43,  421,  171,  ..., 1046,  523, 3994],
        [ 763,   28,   35,  ...,   44,   53, 6461],
        ...,
        [   1,    1,    1,  ...,    1,    1,    1],
        [   1,    1,    1,  ...,    1,    1,    1],
        [   1,    1,    1,  ...,    1,    1,    1]]), 'tgt_ids': tensor([[   2,    2,    2,  ...,    2,    2,    2],
        [  45,    4, 1166,  ...,    4,    4,    4],
        [5877,   64,   11,  ...,  445,  629,   18],
        ...,
        [   1,    1,    1,  ...,    1,    1,    1],
        [   1,    1,    1,  ...,    1,    1,    1],
        [   1,    1,    1,  ...,    1,    1,    1]])}


In [59]:
# BLEU score calculation
bleu = evaluate.load("bleu")

def calculate_bleu(data_loader, model, src_vocab, trg_vocab, device):
    targets, predictions = [], []
    for example in data_loader:
        src_ids, trg_ids = example["src_ids"], example["tgt_ids"]

        # Iterate over each item in the batch
        for src, trg in zip(src_ids, trg_ids):
            pred_trg = translate_sentence(src, src_vocab, trg_vocab, model, device)
            targets.append([trg.tolist()])  # Convert target tensor to list
            predictions.append(pred_trg)

    # Joining target tokens into a single string
    targets = [[" ".join([trg_vocab.lookup_token(idx) for idx in target[0]])] for target in targets]
    return bleu.compute(predictions=predictions, references=targets)

In [66]:
# Calculate BLEU score
test_bleu = calculate_bleu(test_data_loader, model, src_vocab, tgt_vocab, device)
print(f"Test BLEU score: {test_bleu['bleu']:.3f}")

Test BLEU score: 0.000


In [65]:
test_bleu

{'bleu': 3.6831847080089524e-05,
 'precisions': [0.12900944912258147,
  0.01279359664086078,
  0.0021437663294700876,
  6.84369011771147e-05],
 'brevity_penalty': 0.009336906496485076,
 'length_ratio': 0.17624933440583684,
 'translation_length': 15557,
 'reference_length': 88267}

In [63]:
def translate(sentence):
    return translate_sentence(sentence, src_vocab,tgt_vocab, model, device)

In [67]:
sentence = "থুম কা হেনবনা অঙাং অদুদা শাথীনা শোকহনবা য়াই "
print(translate(sentence))

the is of the and and and and and .
